In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import ast
import json
import re
from datetime import datetime

# ==========================================
# 1. CẤU HÌNH & LOAD DATA
# ==========================================
df = pd.read_csv('labeled_dataset.csv')

def fix_path(path):
    # Chuẩn hóa đường dẫn ảnh
    if pd.isna(path): return ""
    return path.replace('\\', '/').replace('../', '')

# Hàm xử lý Tuổi và Giới tính từ 2 cột ketqua_Female/Male
# Logic: Cột nào có dữ liệu (VD: "51 tuổi") thì là giới tính đó
def extract_demographics(row):
    age = 0
    gender = 0 # 0: Male, 1: Female (Mặc định)
    
    # Check cột Female
    if pd.notna(row['ketqua_Female']) and str(row['ketqua_Female']).strip() != "":
        gender = 1 # Female
        age_str = str(row['ketqua_Female'])
    # Check cột Male
    elif pd.notna(row['ketqua_Male']) and str(row['ketqua_Male']).strip() != "":
        gender = 0 # Male
        age_str = str(row['ketqua_Male'])
    else:
        # Nếu cả 2 cột text đều rỗng, thử lấy từ birth_year
        current_year = datetime.now().year
        birth_year = row.get('birth_year', row.get('ketqua_calc_birth_year', current_year))
        age = current_year - birth_year if pd.notna(birth_year) else 0
        return pd.Series([age, 0]) # Mặc định Male nếu ko rõ

    # Parse số tuổi từ chuỗi "51 tuổi"
    try:
        age = int(re.search(r'\d+', age_str).group())
    except:
        # Fallback nếu parse lỗi: dùng birth_year
        current_year = datetime.now().year
        birth_year = row.get('birth_year', row.get('ketqua_calc_birth_year', current_year))
        age = current_year - birth_year if pd.notna(birth_year) else 0
        
    return pd.Series([age, gender])

print(">>> Đang xử lý dữ liệu...")

# ==========================================
# 2. FEATURE ENGINEERING (Tạo cột Age, Sex)
# ==========================================
# Tạo 2 cột mới từ dữ liệu thô
df[['Age', 'Sex']] = df.apply(extract_demographics, axis=1)

# Danh sách cột CHUẨN cần giữ lại cho Model
keep_cols = [
    'ketqua_ID',        # ID để gom nhóm (Study ID)
    'filepath',         # INPUT 1: Ảnh (Visual)
    'ketqua_Diagnose',  # INPUT 2: Chỉ định lâm sàng (Text Context)
    'xray_type',        # INPUT 3: Vị trí chụp (Body Part)
    'Age',              # INPUT 4: Tuổi
    'Sex',              # INPUT 5: Giới tính
    'clean_result',     # OUTPUT 1: Báo cáo kết quả (Ground Truth Text)
    'label'             # OUTPUT 2: Nhãn bệnh (Ground Truth Class)
]

df_clean = df[keep_cols].copy()
df_clean['filepath'] = df_clean['filepath'].apply(fix_path)

# Chuyển label string sang list thật (nếu cần)
df_clean['label'] = df_clean['label'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# ==========================================
# 3. GROUPING (Gom nhiều ảnh của 1 ca)
# ==========================================
print(">>> Đang gom nhóm theo Study ID...")

df_grouped = df_clean.groupby('ketqua_ID').agg({
    'filepath': list,              # Gom ảnh thành list: ['img1.jpg', 'img2.jpg']
    'ketqua_Diagnose': 'first',    # Lấy chỉ định đầu tiên
    'xray_type': 'first',          # Lấy loại chụp
    'Age': 'max',                  # Lấy tuổi (max cho chắc)
    'Sex': 'first',                # Lấy giới tính
    'clean_result': 'first',       # Lấy kết quả text
    'label': 'first'               # Lấy nhãn
}).reset_index()

# Lọc bỏ các ca không có file ảnh hoặc nhãn lỗi
df_grouped = df_grouped[df_grouped['filepath'].map(len) > 0]

print(f"Tổng số ca khám (Study) sau khi gom: {len(df_grouped)}")

# ==========================================
# 4. CLASS MAPPING & SPLITTING
# ==========================================
# Tạo từ điển nhãn
all_labels = set()
for labels in df_grouped['label']:
    if isinstance(labels, list):
        all_labels.update(labels)

# Loại bỏ nhãn 'UNKNOWN' hoặc 'OTHER' khỏi mapping nếu bro không muốn train 2 nhãn này
# (nhưng giữ lại trong data để test cũng được)
if 'UNKNOWN' in all_labels: all_labels.remove('UNKNOWN')
# if 'OTHER' in all_labels: all_labels.remove('OTHER') # Tùy bro quyết định

label2id = {label: i for i, label in enumerate(sorted(list(all_labels)))}
print(">>> Class Mapping:", label2id)

# Lưu mapping
with open('class_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(label2id, f, ensure_ascii=False, indent=4)

# Chia tập Train/Val/Test (70/10/20)
# Dùng label đầu tiên để stratify (cân bằng nhãn)
stratify_col = df_grouped['label'].apply(lambda x: x[0] if len(x) > 0 and x[0] in label2id else "OTHER")

train_val_df, test_df = train_test_split(df_grouped, test_size=0.2, random_state=42, stratify=stratify_col)

# Cập nhật stratify cho vòng 2
stratify_col_remain = train_val_df['label'].apply(lambda x: x[0] if len(x) > 0 and x[0] in label2id else "OTHER")
train_df, val_df = train_test_split(train_val_df, test_size=0.125, random_state=42, stratify=stratify_col_remain)

print(f">>> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# ==========================================
# 5. LƯU FILE
# ==========================================
train_df.to_csv('dataset_train.csv', index=False)
val_df.to_csv('dataset_val.csv', index=False)
test_df.to_csv('dataset_test.csv', index=False)

print(">>> Xong! Đã có dataset_train.csv, dataset_val.csv, dataset_test.csv chuẩn chỉ.")

>>> Đang xử lý dữ liệu...
>>> Đang gom nhóm theo Study ID...
Tổng số ca khám (Study) sau khi gom: 19125
>>> Class Mapping: {'BRONCHITIS': 0, 'CARDIOMEGALY': 1, 'DEGENERATION': 2, 'FRACTURE': 3, 'HARDWARE/SURGERY': 4, 'NORMAL': 5, 'OTHER': 6, 'PLEURAL_EFFUSION': 7, 'PNEUMONIA/INFILTRATION': 8, 'SINUSITIS': 9, 'SOFT_TISSUE': 10, 'SPINAL_ALIGNMENT': 11, 'SPONDYLOSIS/DISC': 12, 'TUBERCULOSIS': 13}
>>> Train: 13387 | Val: 1913 | Test: 3825
>>> Xong! Đã có dataset_train.csv, dataset_val.csv, dataset_test.csv chuẩn chỉ.
